# Import

In [1]:
!pip -q install transformers accelerate datasets sentencepiece

# Model

In [2]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# 더 가볍게 시작하려면:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

print("Loaded:", MODEL_NAME)
print("Device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


# Util Func

In [3]:
import re

def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def clean_answer(text: str) -> str:
    text = text.strip()
    # 첫 줄만 사용
    text = text.split("\n")[0].strip()
    # 코드블록 같은 이상 출력 잘라내기
    text = text.split("```")[0].strip()
    return text

def generate_text(prompt: str, max_new_tokens: int = 128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # temperature/top_p/top_k 안 씀
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

## QA Func

### With Context

In [4]:
def ask_qa_with_context(context: str, question: str, max_new_tokens: int = 48):
    prompt = f"""다음 문서를 읽고 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[문서]
{context}

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)



### Without Context

In [5]:
def ask_qa_without_context(question: str, max_new_tokens: int = 48):
    prompt = f"""다음 질문에 답하세요.

규칙:
1. 답만 한 문장으로 짧게 출력
2. 설명하지 말 것
3. 코드블록 출력 금지

[질문]
{question}

[답변]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)
    if "[답변]" in text:
        text = text.split("[답변]")[-1].strip()
    return clean_answer(text)

## Predict Search Func

In [6]:
def predict_need_search(question: str, context: str = "", max_new_tokens: int = 4):
    prompt = f"""당신의 임무는 외부 검색 필요 여부만 판단하는 것입니다.

규칙:
1. 최신 정보, 현재 시점 정보, 자주 바뀌는 사실이면 SEARCH
2. 일반 상식, 오래 변하지 않는 사실이면 NO_SEARCH
3. 반드시 SEARCH 또는 NO_SEARCH 중 하나만 출력
4. 설명 금지

[질문]
{question}

[문맥]
{context if context else "없음"}

[판단]
"""
    text = generate_text(prompt, max_new_tokens=max_new_tokens)

    if "[판단]" in text:
        result = text.split("[판단]")[-1].strip()
    else:
        result = text.strip()

    result = result.split()[0].upper()

    if result == "SEARCH":
        return "SEARCH"
    elif result == "NO_SEARCH":
        return "NO_SEARCH"
    return result

# Sample Test

In [7]:
context = "서울은 대한민국의 수도이며, 대한민국 최대의 도시이다."
question = "대한민국의 수도는 어디인가?"

print("=== 문서 있음 ===")
print(ask_qa_with_context(context, question))

print("\n=== 문서 없음 ===")
print(ask_qa_without_context(question))

print("\n=== SEARCH 판단 ===")
print("Q1:", predict_need_search("대한민국의 수도는 어디인가?"))   # 기대: NO_SEARCH
print("Q2:", predict_need_search("2026년 현재 대한민국 대통령은 누구인가?"))  # 기대: SEARCH
print("Q3:", predict_need_search("오늘 원달러 환율은 얼마인가?"))   # 기대: SEARCH
print("Q4:", predict_need_search("물의 화학식은 무엇인가?"))       # 기대: NO_SEARCH

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== 문서 있음 ===
서울

=== 문서 없음 ===
서울

=== SEARCH 판단 ===
Q1: NO_SEARCH
Q2: SEARCH
Q3: SEARCH
Q4: NO_SEARCH


# DataSet

In [8]:
from datasets import load_dataset

dataset = load_dataset("klue", "mrc", split="validation[:100]")
print(dataset[0].keys())

README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

dict_keys(['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'])


In [23]:

ds = load_dataset("klue", "mrc")

print("=== Dataset Summary ===")
print(ds)

print("\n=== Split Sizes ===")
for split_name, split_data in ds.items():
    print(f"{split_name}: {len(split_data)}")

print("\n=== Column Names ===")
for split_name, split_data in ds.items():
    print(f"{split_name}: {split_data.column_names}")

print("\n=== First Sample (train) ===")
print(ds["train"][0])

=== Dataset Summary ===
DatasetDict({
    train: Dataset({
        features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
        num_rows: 17554
    })
    validation: Dataset({
        features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
        num_rows: 5841
    })
})

=== Split Sizes ===
train: 17554
validation: 5841

=== Column Names ===
train: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers']
validation: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers']

=== First Sample (train) ===
{'title': '제주도 장마 시작 … 중부는 이달 말부터', 'context': '올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도 늦은 이달 말께 장마가 시작될 전망이다.17일 기상청에 따르면 제주도 남쪽 먼바다에 있는 장마전선의 영향으로 이날 제주도 산간 및 내륙지역에 호우주의보가 내려지면서 곳곳에 100㎜에 육박하는 많은 비가 내렸다. 제주의 장마는 평년보다 2~3일, 지난해보다는 

## rough match Func

In [9]:
def rough_match(pred: str, gold: str) -> bool:
    pred_n = normalize_text(pred)
    gold_n = normalize_text(gold)
    return (gold_n in pred_n) or (pred_n in gold_n)

# BaseLine Test With Context

In [10]:
correct = 0
total = 0
results = []

for ex in dataset:
    context = ex["context"]
    question = ex["question"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    pred = ask_qa_with_context(context, question)

    ok = rough_match(pred, gold)
    correct += int(ok)
    total += 1

    results.append({
        "question": question,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

print(f"With context accuracy (rough): {correct}/{total} = {correct/total:.3f}")
results[:3]

With context accuracy (rough): 66/100 = 0.660


[{'question': '말라카이트에서 나온 색깔을 사용한 에디션은?',
  'gold': '뉴 740Li 25주년 에디션',
  'pred': '말라카이트 그린 다크 색상이 적용된 뉴 740Li 25주년 에디션(7대 한정)에 사용되었다. 이 색상은',
  'correct': True},
 {'question': '사고 비행기의 목적지는?',
  'gold': '독일 뒤셀도르프',
  'pred': '독일 뒤셀도르프로',
  'correct': True},
 {'question': '퇴사하고 싶은 회사로 트위터가 순위에 선정된 해는?',
  'gold': '2014년',
  'pred': '2014년',
  'correct': True}]

In [11]:
wrong_cases = [r for r in results if not r["correct"]]
print(f"Wrong cases: {len(wrong_cases)}")

for i, case in enumerate(wrong_cases[:5]):
    print(f"\n[Wrong Case {i+1}]")
    print("Q:", case["question"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Wrong cases: 34

[Wrong Case 1]
Q: 2014년 일하고 싶은 50대 회사 중에서 5위로 선정된 기업은?
Gold: 페이스북
Pred: 링크트인

[Wrong Case 2]
Q: 포브스의 2014년 일하고 싶은 50대 회사 조사에서 5위를 한 기업은?
Gold: 페이스북
Pred: 링크트인

[Wrong Case 3]
Q: 가공하는데 몇 초밖에 걸리지 않는 물질은?
Gold: 실리콘유
Pred: 셀룰로오스의 히드록시기와 반응하여 천 위를 덮어서 방수성을 준다. (몇 초만)

[Wrong Case 4]
Q: 실리콘을 실리콘유로 만들기 위해 거치는 과정은?
Gold: 실리콘
Pred: 실란을 원료로 사용하여 디메틸폴리실록산으로 만든다

[Wrong Case 5]
Q: 성서를 영어로 번역한 인물에게 영향을 받은 사람은?
Gold: 얀 후스
Pred: 존 위클리프


# BaseLine Test Without Context

In [12]:
correct_no_ctx = 0
total_no_ctx = 0
results_no_ctx = []

for ex in dataset:
    question = ex["question"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    pred = ask_qa_without_context(question)

    ok = rough_match(pred, gold)
    correct_no_ctx += int(ok)
    total_no_ctx += 1

    results_no_ctx.append({
        "question": question,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

print(f"Without context accuracy (rough): {correct_no_ctx}/{total_no_ctx} = {correct_no_ctx/total_no_ctx:.3f}")
results_no_ctx[:3]

Without context accuracy (rough): 2/100 = 0.020


[{'question': '말라카이트에서 나온 색깔을 사용한 에디션은?',
  'gold': '뉴 740Li 25주년 에디션',
  'pred': '레드 에디션',
  'correct': False},
 {'question': '사고 비행기의 목적지는?',
  'gold': '독일 뒤셀도르프',
  'pred': '서울',
  'correct': False},
 {'question': '퇴사하고 싶은 회사로 트위터가 순위에 선정된 해는?',
  'gold': '2014년',
  'pred': '2020년 1월 9일',
  'correct': False}]

In [13]:
wrong_cases_no_ctx = [r for r in results_no_ctx if not r["correct"]]
print(f"Wrong cases without context: {len(wrong_cases_no_ctx)}")

for i, case in enumerate(wrong_cases_no_ctx[:5]):
    print(f"\n[Wrong No-Context Case {i+1}]")
    print("Q:", case["question"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Wrong cases without context: 98

[Wrong No-Context Case 1]
Q: 말라카이트에서 나온 색깔을 사용한 에디션은?
Gold: 뉴 740Li 25주년 에디션
Pred: 레드 에디션

[Wrong No-Context Case 2]
Q: 사고 비행기의 목적지는?
Gold: 독일 뒤셀도르프
Pred: 서울

[Wrong No-Context Case 3]
Q: 퇴사하고 싶은 회사로 트위터가 순위에 선정된 해는?
Gold: 2014년
Pred: 2020년 1월 9일

[Wrong No-Context Case 4]
Q: 2014년 일하고 싶은 50대 회사 중에서 5위로 선정된 기업은?
Gold: 페이스북
Pred: 삼성전자

[Wrong No-Context Case 5]
Q: 포브스의 2014년 일하고 싶은 50대 회사 조사에서 5위를 한 기업은?
Gold: 페이스북
Pred: IBM


## PipeLine

In [14]:
def qa_with_search_pipeline(question, context):
    decision = predict_need_search(question, context)

    if decision == "SEARCH":
        pred = ask_qa_with_context(context, question)
    else:
        pred = ask_qa_without_context(question)

    return decision, pred

### PipeLine Test

In [15]:
pipeline_correct = 0
pipeline_results = []

for ex in dataset:
    question = ex["question"]
    context = ex["context"]
    gold_answers = ex["answers"]["text"]
    gold = gold_answers[0] if len(gold_answers) > 0 else ""

    decision, pred = qa_with_search_pipeline(question, context)

    ok = rough_match(pred, gold)
    pipeline_correct += int(ok)

    pipeline_results.append({
        "question": question,
        "decision": decision,
        "gold": gold,
        "pred": pred,
        "correct": ok
    })

pipeline_acc = pipeline_correct / len(dataset)
print(f"Pipeline accuracy: {pipeline_correct}/{len(dataset)} = {pipeline_acc:.3f}")

Pipeline accuracy: 62/100 = 0.620


### PipeLine Errors

In [16]:
pipeline_errors = [r for r in pipeline_results if not r["correct"]]
print(f"Pipeline wrong cases: {len(pipeline_errors)}")

for i, case in enumerate(pipeline_errors[:10]):
    print(f"\n[Pipeline Error {i+1}]")
    print("Q:", case["question"])
    print("Decision:", case["decision"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Pipeline wrong cases: 38

[Pipeline Error 1]
Q: 2014년 일하고 싶은 50대 회사 중에서 5위로 선정된 기업은?
Decision: SEARCH
Gold: 페이스북
Pred: 링크트인

[Pipeline Error 2]
Q: 포브스의 2014년 일하고 싶은 50대 회사 조사에서 5위를 한 기업은?
Decision: SEARCH
Gold: 페이스북
Pred: 링크트인

[Pipeline Error 3]
Q: 가공하는데 몇 초밖에 걸리지 않는 물질은?
Decision: SEARCH
Gold: 실리콘유
Pred: 셀룰로오스의 히드록시기와 반응하여 천 위를 덮어서 방수성을 준다. (몇 초만)

[Pipeline Error 4]
Q: 실리콘을 실리콘유로 만들기 위해 거치는 과정은?
Decision: SEARCH
Gold: 실리콘
Pred: 실란을 원료로 사용하여 디메틸폴리실록산으로 만든다

[Pipeline Error 5]
Q: 성서를 영어로 번역한 인물에게 영향을 받은 사람은?
Decision: SEARCH
Gold: 얀 후스
Pred: 존 위클리프

[Pipeline Error 6]
Q: 법무법인 한결이 세워진 지 10년차에 합병된 업체의 이름은?
Decision: SEARCH
Gold: 법무법인 내일
Pred: 한살림

[Pipeline Error 7]
Q: 한국예술영재교육원 원장이 현재 학생을 가르치는 곳은?
Decision: SEARCH
Gold: 한국예술종합학교 음악원
Pred: 한예종 부설 한국예술영재교육원

[Pipeline Error 8]
Q: 허벅지까지 내려오는 미들 기장의 다운 자켓은 이름은?
Decision: NO_SEARCH
Gold: ‘로사다운자켓’
Pred: Mid-Calf Length Down Jacket

[Pipeline Error 9]
Q: 데카메론에서 마을의 모든 부자들에게 돈을 달라고 하는 인물은?
Decision: SEARCH
Gold: 걸인
Pred: 없음

[Pipeline Error 10

In [22]:
under_search = []
qa_failure = []
over_search = []

for r in pipeline_results:
    if not r["correct"]:
        if r["decision"] == "NO_SEARCH":
            under_search.append(r)
        elif r["decision"] == "SEARCH":
            qa_failure.append(r)

print("UNDER_SEARCH:", len(under_search))
print("QA_FAILURE:", len(qa_failure))

UNDER_SEARCH: 5
QA_FAILURE: 33


# SEARCH / NO_SEARCH baseline용 소규모 수작업 평가

In [17]:
search_eval = [
    # ----------------------------
    # NO_SEARCH: 일반 상식 / 고정 지식 / 문맥으로 바로 답 가능
    # ----------------------------
    {"question": "대한민국의 수도는 어디인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "물의 화학식은 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "조선의 건국자는 누구인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "세종대왕은 어느 왕조의 왕인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "삼권분립이란 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "광합성은 무엇을 만드는 과정인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "대한민국의 국기는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "지구는 태양계에서 몇 번째 행성인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "피타고라스 정리는 어떤 관계를 설명하는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "한글을 창제한 왕은 누구인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "프랑스의 수도는 어디인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "뉴턴의 제2법칙은 무엇을 설명하는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "DNA는 무엇의 약자인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "대한민국의 국화는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "아인슈타인이 제안한 대표 이론은 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "태양은 어느 은하에 속해 있는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "원주율을 보통 어떤 기호로 나타내는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "대한민국의 공식 언어는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "미국의 수도는 어디인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "산소의 원소기호는 무엇인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "이순신은 어느 시대의 인물인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "컴퓨터에서 CPU는 무엇의 약자인가?", "context": "", "label": "NO_SEARCH"},
    {"question": "태평양은 대서양보다 넓은가?", "context": "", "label": "NO_SEARCH"},
    {"question": "조류는 알을 낳는가?", "context": "", "label": "NO_SEARCH"},
    {"question": "이 문서에서 저자가 주장한 핵심은 무엇인가?", "context": "본 논문은 small LLM에서 search policy distillation 가능성을 탐구한다.", "label": "NO_SEARCH"},

    # ----------------------------
    # SEARCH: 최신 정보 / 변동 정보 / 외부 확인 필요
    # ----------------------------
    {"question": "2026년 현재 대한민국 대통령은 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 원달러 환율은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 주 삼성전자 종가는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 서울 날씨는 어떤가?", "context": "", "label": "SEARCH"},
    {"question": "2025년 노벨문학상 수상자는 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 시즌 EPL 우승팀은 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 비트코인 가격은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 금 가격은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 주 코스피 지수는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 뉴욕 증시 다우지수는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 애플 시가총액은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 달 한국은행 기준금리는 몇 퍼센트인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 서울의 미세먼지 농도는 어떤가?", "context": "", "label": "SEARCH"},
    {"question": "올해 아카데미 작품상 수상작은 무엇인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 테슬라 주가는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 S&P500 지수는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "이번 주 국제 유가(WTI)는 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "지금 한국의 실업률은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 엔비디아 CEO는 누구인가?", "context": "", "label": "SEARCH"},
    {"question": "올해 UEFA 챔피언스리그 우승팀은 누구인가?", "context": "", "label": "SEARCH"},

    # ----------------------------
    # 경계/애매한 케이스: search 필요성 판단이 헷갈릴 수 있는 문제
    # ----------------------------
    {"question": "대한민국의 인구는 몇 명인가?", "context": "", "label": "SEARCH"},
    {"question": "서울의 면적은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "현재 가장 높은 빌딩은 무엇인가?", "context": "", "label": "SEARCH"},
    {"question": "대한민국의 최저임금은 얼마인가?", "context": "", "label": "SEARCH"},
    {"question": "오늘 날짜는 무엇인가?", "context": "", "label": "SEARCH"},
]

In [18]:
pred_correct = 0
search_results = []

for ex in search_eval:
    pred = predict_need_search(ex["question"], ex["context"])
    ok = pred == ex["label"]
    pred_correct += int(ok)

    search_results.append({
        "question": ex["question"],
        "gold": ex["label"],
        "pred": pred,
        "correct": ok
    })

    print({
        "question": ex["question"],
        "gold": ex["label"],
        "pred": pred,
        "correct": ok
    })

print(f"\nSearch decision accuracy: {pred_correct}/{len(search_eval)} = {pred_correct/len(search_eval):.3f}")

{'question': '대한민국의 수도는 어디인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '물의 화학식은 무엇인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '조선의 건국자는 누구인가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '세종대왕은 어느 왕조의 왕인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '삼권분립이란 무엇인가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '광합성은 무엇을 만드는 과정인가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '대한민국의 국기는 무엇인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '지구는 태양계에서 몇 번째 행성인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '피타고라스 정리는 어떤 관계를 설명하는가?', 'gold': 'NO_SEARCH', 'pred': 'SEARCH', 'correct': False}
{'question': '한글을 창제한 왕은 누구인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': True}
{'question': '프랑스의 수도는 어디인가?', 'gold': 'NO_SEARCH', 'pred': 'NO_SEARCH', 'correct': T

In [19]:
wrong_search = [r for r in search_results if not r["correct"]]
print(f"Wrong search-decision cases: {len(wrong_search)}")

for i, case in enumerate(wrong_search):
    print(f"\n[Wrong Search Case {i+1}]")
    print("Q:", case["question"])
    print("Gold:", case["gold"])
    print("Pred:", case["pred"])

Wrong search-decision cases: 11

[Wrong Search Case 1]
Q: 조선의 건국자는 누구인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 2]
Q: 삼권분립이란 무엇인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 3]
Q: 광합성은 무엇을 만드는 과정인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 4]
Q: 피타고라스 정리는 어떤 관계를 설명하는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 5]
Q: 뉴턴의 제2법칙은 무엇을 설명하는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 6]
Q: 아인슈타인이 제안한 대표 이론은 무엇인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 7]
Q: 태양은 어느 은하에 속해 있는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 8]
Q: 원주율을 보통 어떤 기호로 나타내는가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 9]
Q: 미국의 수도는 어디인가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 10]
Q: 태평양은 대서양보다 넓은가?
Gold: NO_SEARCH
Pred: SEARCH

[Wrong Search Case 11]
Q: 이 문서에서 저자가 주장한 핵심은 무엇인가?
Gold: NO_SEARCH
Pred: SEARCH


# Save

In [20]:
import json

with open("qa_with_context_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open("qa_without_context_results.json", "w", encoding="utf-8") as f:
    json.dump(results_no_ctx, f, ensure_ascii=False, indent=2)

print("Saved result files.")

Saved result files.
